#  Arm Assembly Primer : Where arrays are
# 1. Five ways to get an array
* C/C++ offers several ways to get memory for an array of 100 `long`s
  * a local array (on the _stack_)
  * a global array
  * `malloc` (C), `new` (C++), and `mmap` (an operating system call) (on the _heap_ or in separately mapped memory)
* In every case, what the program gets is an address; let's see how the address is obtained. `use` is a function defined elsewhere that receives the array as a pointer


In [ ]:
%%writefile mem.cc
#include <stdlib.h>
#include <sys/mman.h>
long use(long * p);          /* defined elsewhere */

long g[100];                 /* global array */

long local_array() {
  long a[100];
  return use(a);
}

long global_array() {
  return use(g);
}

long malloc_array() {
  long * a = (long *)malloc(sizeof(long) * 100);
  return use(a);
}

long new_array() {
  long * a = new long[100];
  return use(a);
}

long mmap_array() {
  long * a = (long *)mmap(0, sizeof(long) * 100, PROT_READ | PROT_WRITE,
                          MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
  return use(a);
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O1 -S mem.cc
c++filt < mem.s | grep -vE '^\s+\.|^\s*//|^\.Lfunc'

* (`c++filt` shows `_Z3usePl` as `use(long*)`, etc.)
* You will see something like this

```
local_array():
	stp	x29, x30, [sp, #-32]!
	str	x28, [sp, #16]
	mov	x29, sp
	sub	sp, sp, #800         // make room for 100 longs (800 bytes) on the stack
	mov	x0, sp               // the address of the array = sp
	bl	use(long*)
	add	sp, sp, #800         // give the room back
	ldr	x28, [sp, #16]
	ldp	x29, x30, [sp], #32
	ret
global_array():
	adrp	x0, g                // the address of g (in two steps)
	add	x0, x0, :lo12:g
	b	use(long*)
malloc_array():
	...
	mov	w0, #800             // the argument of malloc: 800 bytes
	bl	malloc               // the address is returned in x0
	...
	b	use(long*)
new_array():
	...
	mov	w0, #800
	bl	operator new[](unsigned long)
	...
	b	use(long*)
mmap_array():
	...
	mov	x0, xzr              // the six arguments of mmap
	mov	w1, #800
	mov	w2, #3
	mov	w3, #34
	mov	w4, #-1
	mov	x5, xzr
	bl	mmap
	...
	b	use(long*)
```

* In all five, the array ends up as an address in `x0`, passed to `use` in exactly the same way; `use` cannot tell where the array came from
* <font color="blue">Local array</font>: the stack is a region of memory, and `sp` points to its top. A function makes room for its local array just by subtracting from `sp` (`sub sp, sp, #800`), and the array's address is `sp` (or `sp` plus a small offset). It gives the room back by adding to `sp` before returning, which is why a local array disappears when the function returns
* <font color="blue">Global array</font>: `g` has a fixed address decided when the program is loaded; `adrp` + `add` computes it (an instruction cannot hold a 64-bit address, so it is computed in two parts)
* <font color="blue">`malloc`, `new`, `mmap`</font>: they are just functions; they find free memory and return its address in `x0`. `new long[100]` calls a function named `operator new[]` (`_Znam`), which does essentially what `malloc` does. `sizeof(long) * 100` was computed by the compiler (`#800`)

# 2. Stack protector
* Many compilers (e.g., the default `gcc` and `clang` of Ubuntu and some other Linux distributions) add extra instructions to functions having local arrays, to detect a common kind of bug (writing beyond the end of a local array). The compilers we use on Miyabi do not do so by default, but you may meet it elsewhere; the option `-fstack-protector-strong` turns it on


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O1 -S -fstack-protector-strong mem.cc -o mem_sp.s
c++filt < mem_sp.s | grep -vE '^\s+\.|^\s*//|^\.Lfunc'

* `local_array` now looks like this

```
local_array():
	stp	x29, x30, [sp, #-32]!
	str	x28, [sp, #16]
	mov	x29, sp
	sub	sp, sp, #816                          // 16 bytes more than the array
	adrp	x8, :got:__stack_chk_guard
	add	x0, sp, #8                            // the array starts at sp + 8
	ldr	x8, [x8, :got_lo12:__stack_chk_guard]
	ldr	x8, [x8]                              // x8 = a secret value (the "guard")
	stur	x8, [x29, #-8]                        // put it right after the array
	bl	use(long*)
	adrp	x8, :got:__stack_chk_guard
	ldr	x8, [x8, :got_lo12:__stack_chk_guard]
	ldur	x9, [x29, #-8]                        // the value right after the array
	ldr	x8, [x8]                              // the guard
	cmp	x8, x9
	b.ne	.LBB0_2                               // different? the array overflowed
	add	sp, sp, #816
	ldr	x28, [sp, #16]
	ldp	x29, x30, [sp], #32
	ret
.LBB0_2:
	bl	__stack_chk_fail                      // report the error and abort
```

* The function puts a secret value right after the array before calling `use`, and checks it is still there afterwards; if `use` wrote beyond the end of the array, the value has changed, and the program is aborted
* This is useful for security, but clutters the assembly code. If your compiler adds it by default and you want to read the assembly code more easily, give `-fno-stack-protector`

# 3. They are all numbers
* Let's print the addresses at runtime


In [ ]:
%%writefile mem_addr.cc
#include <stdio.h>
#include <stdlib.h>
#include <sys/mman.h>

long g[100];

int main() {
  long a[100];
  long * m = (long *)malloc(sizeof(long) * 100);
  long * n = new long[100];
  long * p = (long *)mmap(0, sizeof(long) * 100, PROT_READ | PROT_WRITE,
                          MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
  printf("local  : %p\n", (void *)a);
  printf("global : %p\n", (void *)g);
  printf("malloc : %p\n", (void *)m);
  printf("new    : %p\n", (void *)n);
  printf("mmap   : %p\n", (void *)p);
  a[1] = g[1] = m[1] = n[1] = p[1] = 1;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O1 mem_addr.cc -o mem_addr
./mem_addr

* They are just numbers in different ranges: the global array near the program itself, `malloc` and `new` in the area after it (the heap), `mmap` and the stack in very high addresses
* Whatever its origin, an array is accessed with the same `ldr`/`str` instructions we saw in pd09_access

# 4. Summary
* local arrays: room made on the stack by moving `sp`; the address is computed from `sp`
* global arrays: a fixed address computed by `adrp` + `add`
* `malloc`, `new`, `mmap`: functions returning an address in `x0`
* in every case, an array is an address, and a pointer is that address